# Análisis de textos médicos

Iago Grandal Del Río, Claudia Vidal Otero

In [ ]:
!pip install pandas
!pip install fastparquet
!pip install "llama-cpp-python>=0.2.0" huggingface-hub
!pip install datasets

In [ ]:
from datasets import load_dataset
import pandas as pd
from llama_cpp import Llama, LlamaGrammar

Cargamos los datos de wikidisease del dataset ClinText

In [ ]:
dataset = load_dataset("IIC/ClinText-SP")
filtered = dataset["train"].filter(lambda x: x["source"] == "wikidisease")

df = filtered.to_pandas()
print(df.head())
print(len(filtered))

texts = df["text"].head(5).tolist()



                                                text       source
0  **Enfermedad**: Hiperparatiroidismo\n**Abstrac...  wikidisease
1  **Enfermedad**: Alveolitis alérgica extrínseca...  wikidisease
2  **Enfermedad**: Hipertensión arterial\n**Abstr...  wikidisease
3  **Enfermedad**: Hipertrigliceridemia\n**Abstra...  wikidisease
4  **Enfermedad**: Hipocalcemia\n**Abstract**\nLa...  wikidisease
945


Definimos una gramática para el forzado

In [11]:
grammar_text = r"""
root ::= "{ \"disease\": " string ", \"abstract\": " string ", \"tratamiento\": " string " }"

string ::= "\"" chars "\""
chars ::= char*
char ::= [^\"]
"""
grammar = LlamaGrammar.from_string(grammar_text)


Llamamos al modelo

In [ ]:

llm = Llama.from_pretrained(
    repo_id="TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF",
    filename="tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
    n_ctx=2048,
    n_gpu_layers=0,
    verbose=False,
    seed=1234,
)


c:\Users\108057\OneDrive - Universidade da Coruña\FIC\AÑO4\BM\MEDICINA\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Creamos una función con un prompt que ayude al forzado en la extracción

In [13]:

def extract_info(text):
    prompt = f"""
        Extrae información clínica del texto y devuelve SOLO un objeto JSON válido.

        El JSON debe tener EXACTAMENTE estas claves y en este orden:
        1) disease
        2) abstract
        3) tratamiento

        Reglas:
        - Todos los valores deben ser strings entre comillas dobles.
        - No incluyas ninguna clave adicional.
        - No incluyas texto fuera del JSON.
        - Si falta información, usa:
          "desconocido"

        Texto:
        {text}
        """

    out = llm(
        prompt,
        max_tokens=1024,
        temperature=0,
        repeat_penalty=1.2,
        grammar=grammar,
        stop=["}\n"]
    )

    return out["choices"][0]["text"].strip()

Extraemos la información y mostramos los resultados

In [14]:
results = []

for t in texts:
    result = extract_info(t)
    results.append(result)
    print(result)

{ "disease": "Hiperparaatiroidismo", "abstract": "El hiperparaatirodismo es una alteración que consiste en que las glándulas paratiroideaas segregan mayor cantidad de hornoa paratiroide, reguladora del calcio, magnesio y fósforo en la sangre y hueso.", "tratamiento": "Consistirá en la administración de fosfaotos, hornomonas, si hubiera tumor hipersecretante, extirpación quirúrgica." }
{ "disease": "Alveolitis alérgica extrínsecamente", "abstract": "La Alveolitis alérgica o Neumonitis por hipersensibilidad, es un síndrome que comprende una variedad de enfermedades caracterizada por la respuesta pulmonar a la inhalación repetida de una variedad de polvos orgánico u hongoz, causando una respuesta inmunitaria. Existen tres formas clínicas:", "tratamiento": "El mejor tratamiento es evitar el alérgeno provocador, ya que la exposición crónica puede causa daño permanente y la enfermedad aguda a menudo se auto-limita." }
{ "disease": "HTA", "abstract": "Hipertensión arterial (HTA) es una enferm